## LIBRERIAS PARA QUE EL CODIGO FUNCIONE CORRECTAMENTE

In [27]:
!pip install psycopg2-binary
!pip install pandas

## GENERACION DE LA REPORTERIA SOLICITADA

In [38]:
import pandas as pd
import psycopg2
# Conexión a la base de datos
conn = psycopg2.connect(
    dbname="postgres", 
    user="postgres", 
    password="postgres", 
    host="localhost", 
    port="5432"
)

# Cursor para interactuar con la base de datos
cursor = conn.cursor()

# 1. Consultar los Top 10 Productos más vendidos
consulta_productos = """
SELECT p.nombre_producto,
       SUM(hv.cantidad) AS cantidad_vendida
FROM hecho_ventas hv
JOIN dim_producto p ON hv.id_producto = p.id_producto
GROUP BY p.id_producto, p.nombre_producto
ORDER BY cantidad_vendida DESC
LIMIT 10;
"""
df_top_10_productos = pd.read_sql(consulta_productos, conn)
print(df_top_10_productos)

                  nombre_producto  cantidad_vendida
0  AURICULARES INALAMBRICOS NTUNE                 4
1       RADIO DE AUTO PIONEER DEH                 4
2       SISTEMA DE SONIDO JBL CAR                 3
3     CARGADOR USB RAPIDO   COCHE                 3
4       PANTALLA TACTIL 7"   AUTO                 3
5     SOPORTE MAGNETICO   CELULAR                 2
6     SMARTWATCH XIAOMI MI BAND 7                 2
7               ALTAVOZ SONY XB33                 2
8               LAPTOP HP ENVY 13                 2
9         CAMARA DE REVERSA NTECH                 2


/tmp/ipykernel_1554/3763599344.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_10_productos = pd.read_sql(consulta_productos, conn)


In [39]:
# 2. Consultar los Top 5 Clientes con el mayor número de pedidos
consulta_clientes = """
SELECT c.nombres || ' ' || c.apellidos AS cliente,
       COUNT(hv.id_cliente) AS numero_pedidos
FROM hecho_ventas hv
JOIN dim_cliente c ON hv.id_cliente = c.id_cliente
GROUP BY c.id_cliente, c.nombres, c.apellidos
ORDER BY numero_pedidos DESC
LIMIT 5;
"""
df_top_5_clientes = pd.read_sql(consulta_clientes, conn)
print(df_top_5_clientes)

                   cliente  numero_pedidos
0     FERNANDA PEREZ LOPEZ               6
1  MARIA JULIA GOMEZ NUNEZ               4
2           CARLOS RAMIREZ               4
3  JOSE ANDRES MUNOZ PEREZ               4
4       ANA LUCIA ZAMBRANO               2


/tmp/ipykernel_1554/1611583290.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_5_clientes = pd.read_sql(consulta_clientes, conn)


In [40]:
# 3. Top 5 Corresponsales con el mayor número de pedidos
consulta_corresponsales = """
SELECT cor.nombre_corresponsal,
       COUNT(hv.id_corresponsal) AS numero_pedidos
FROM hecho_ventas hv
JOIN dim_corresponsal cor ON hv.id_corresponsal = cor.id_corresponsal
GROUP BY cor.id_corresponsal, cor.nombre_corresponsal
ORDER BY numero_pedidos DESC
LIMIT 5;
"""
df_top_5_corresponsales = pd.read_sql(consulta_corresponsales, conn)
print(df_top_5_corresponsales)

          nombre_corresponsal  numero_pedidos
0  MULTISERVICIOS LOPEZ PEREZ               6
1                   AUTO ZONA               4
2                 AUDIO MUNDO               4
3               ELECTRO HOGAR               4
4             TECNOLOGIA NANA               4


/tmp/ipykernel_1554/2841871516.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_top_5_corresponsales = pd.read_sql(consulta_corresponsales, conn)


In [41]:
# 4. Total de pagos diario y mensual por productos
consulta_pagos_diarios_mensuales_productos = """
SELECT p.nombre_producto,
       TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM-DD') AS fecha_diaria,
       SUM(hv.total_pagado) AS total_pagado_diario,
       TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM') AS fecha_mensual,
       -- Subconsulta para calcular el total mensual por producto
       (SELECT SUM(hv2.total_pagado)
        FROM hecho_ventas hv2
        JOIN dim_tiempo t2 ON hv2.id_fecha = t2.id_fecha
        WHERE p.id_producto = hv2.id_producto
        AND TO_CHAR(CAST(t2.fecha AS DATE), 'YYYY-MM') = TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM')
       ) AS total_pagado_mensual
FROM hecho_ventas hv
JOIN dim_producto p ON hv.id_producto = p.id_producto
JOIN dim_tiempo t ON hv.id_fecha = t.id_fecha
GROUP BY p.id_producto, p.nombre_producto, t.fecha
ORDER BY t.fecha;
"""
df_pagos_productos = pd.read_sql(consulta_pagos_diarios_mensuales_productos, conn)
df_pagos_productos.sort_values(by=["nombre_producto", "fecha_diaria"], inplace=True)
print(df_pagos_productos)

                   nombre_producto fecha_diaria  total_pagado_diario  \
3                ALTAVOZ SONY XB33   2025-04-29               298.26   
2   AURICULARES INALAMBRICOS NTUNE   2025-04-29               130.00   
15  AURICULARES INALAMBRICOS NTUNE   2025-04-30               390.00   
6          CAMARA DE REVERSA NTECH   2025-04-29                59.99   
18         CAMARA DE REVERSA NTECH   2025-04-30                59.99   
11     CARGADOR USB RAPIDO   COCHE   2025-04-29                46.65   
1                LAPTOP HP ENVY 13   2025-04-29              1799.02   
12     LUCES LED INTERIORES   AUTO   2025-04-29               180.00   
13          MICROFONO USB C NSOUND   2025-04-29               130.00   
5        PANTALLA TACTIL 7"   AUTO   2025-04-29               200.00   
17       PANTALLA TACTIL 7"   AUTO   2025-04-30               400.00   
4        RADIO DE AUTO PIONEER DEH   2025-04-29                90.00   
16       RADIO DE AUTO PIONEER DEH   2025-04-30               27

/tmp/ipykernel_1554/1906558883.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_productos = pd.read_sql(consulta_pagos_diarios_mensuales_productos, conn)


In [42]:
# 5. Total de pagos diario y mensual por clientes
consulta_pagos_diarios_mensuales_clientes = """
SELECT 
    c.nombres || ' ' || c.apellidos AS nombre_cliente,
    TO_CHAR(t.fecha::DATE, 'YYYY-MM-DD') AS fecha_diaria,
    SUM(hv.total_pagado) AS total_pagado_diario,
    TO_CHAR(t.fecha::DATE, 'YYYY-MM') AS fecha_mensual,
    
    (
        SELECT SUM(hv2.total_pagado)
        FROM hecho_ventas hv2
        JOIN dim_tiempo t2 ON hv2.id_fecha = t2.id_fecha
        WHERE c.id_cliente = hv2.id_cliente
        AND TO_CHAR(CAST(t2.fecha AS DATE), 'YYYY-MM') = TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM')
    ) AS total_pagado_mensual

FROM hecho_ventas hv
JOIN dim_cliente c ON hv.id_cliente = c.id_cliente
JOIN dim_tiempo t ON hv.id_fecha = t.id_fecha
GROUP BY c.id_cliente, c.nombres, c.apellidos, t.fecha
ORDER BY nombre_cliente, fecha_diaria;
"""
df_pagos_diarios_mensuales_clientes = pd.read_sql(consulta_pagos_diarios_mensuales_clientes, conn)
df_pagos_diarios_mensuales_clientes.sort_values(by=["nombre_cliente", "fecha_diaria"], inplace=True)
print(df_pagos_diarios_mensuales_clientes)

             nombre_cliente fecha_diaria  total_pagado_diario fecha_mensual  \
0        ANA LUCIA ZAMBRANO   2025-04-29               310.00       2025-04   
1            CARLOS RAMIREZ   2025-04-29               986.13       2025-04   
2      FERNANDA PEREZ LOPEZ   2025-04-30              1299.99       2025-04   
3   JOSE ANDRES MUNOZ PEREZ   2025-04-29              2777.28       2025-04   
4  LUIS MIGUEL TORRES NAHUI   2025-04-30               848.26       2025-04   
5   MARIA JULIA GOMEZ NUNEZ   2025-04-29               389.99       2025-04   

   total_pagado_mensual  
0                310.00  
1                986.13  
2               1299.99  
3               2777.28  
4                848.26  
5                389.99  


/tmp/ipykernel_1554/3809695329.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_diarios_mensuales_clientes = pd.read_sql(consulta_pagos_diarios_mensuales_clientes, conn)


In [43]:
# 6. Total de pagos diario y mensual por corresponsal
consulta_pagos_diarios_mensuales_corresponsal = """
SELECT 
    co.nombre_corresponsal AS nombre_corresponsal,
    TO_CHAR(t.fecha::DATE, 'YYYY-MM-DD') AS fecha_diaria,
    SUM(hv.total_pagado) AS total_pagado_diario,
    TO_CHAR(t.fecha::DATE, 'YYYY-MM') AS fecha_mensual,
    
    (
        SELECT SUM(hv2.total_pagado)
        FROM hecho_ventas hv2
        JOIN dim_tiempo t2 ON hv2.id_fecha = t2.id_fecha
        WHERE co.id_corresponsal = hv2.id_corresponsal
        AND TO_CHAR(CAST(t2.fecha AS DATE), 'YYYY-MM') = TO_CHAR(CAST(t.fecha AS DATE), 'YYYY-MM')
    ) AS total_pagado_mensual

FROM hecho_ventas hv
JOIN dim_corresponsal co ON hv.id_corresponsal = co.id_corresponsal
JOIN dim_tiempo t ON hv.id_fecha = t.id_fecha
GROUP BY co.id_corresponsal, co.nombre_corresponsal, t.fecha
ORDER BY nombre_corresponsal, fecha_diaria;
"""
df_pagos_diarios_mensuales_corresponsal = pd.read_sql(consulta_pagos_diarios_mensuales_corresponsal, conn)
df_pagos_diarios_mensuales_corresponsal.sort_values(by=["nombre_corresponsal", "fecha_diaria"], inplace=True)
print(df_pagos_diarios_mensuales_corresponsal)

          nombre_corresponsal fecha_diaria  total_pagado_diario fecha_mensual  \
0                 AUDIO MUNDO   2025-04-29               738.26       2025-04   
1                   AUTO ZONA   2025-04-29                99.99       2025-04   
2                   AUTO ZONA   2025-04-30               480.00       2025-04   
3               ELECTRO HOGAR   2025-04-29               290.00       2025-04   
4               ELECTRO HOGAR   2025-04-30               848.26       2025-04   
5  MULTISERVICIOS LOPEZ PEREZ   2025-04-29               579.50       2025-04   
6  MULTISERVICIOS LOPEZ PEREZ   2025-04-30               819.99       2025-04   
7             TECNOLOGIA NANA   2025-04-29              2755.65       2025-04   

   total_pagado_mensual  
0                738.26  
1                579.99  
2                579.99  
3               1138.26  
4               1138.26  
5               1399.49  
6               1399.49  
7               2755.65  


/tmp/ipykernel_1554/2718552775.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pagos_diarios_mensuales_corresponsal = pd.read_sql(consulta_pagos_diarios_mensuales_corresponsal, conn)


In [44]:
# 7. Variación diaria, mensual de pedidos por productos y corresponsal
consulta_variacion_diaria_mensual_productos_corresponsal = """
WITH pagos_diarios_mensuales AS (
    SELECT 
        p.id_producto,
        co.id_corresponsal,
        TO_CHAR(t.fecha::DATE, 'YYYY-MM-DD') AS fecha_diaria,
        SUM(hv.total_pagado) AS total_pagado_diario,
        TO_CHAR(t.fecha::DATE, 'YYYY-MM') AS fecha_mensual,
        SUM(hv.total_pagado) AS total_pagado_mensual
    FROM hecho_ventas hv
    JOIN dim_producto p ON hv.id_producto = p.id_producto
    JOIN dim_corresponsal co ON hv.id_corresponsal = co.id_corresponsal
    JOIN dim_tiempo t ON hv.id_fecha = t.id_fecha
    GROUP BY p.id_producto, co.id_corresponsal, t.fecha
)
SELECT
    pr.nombre_producto,
    co.nombre_corresponsal,
    fecha_diaria,
    total_pagado_diario,
    fecha_mensual,
    total_pagado_mensual,
    total_pagado_diario - LAG(total_pagado_diario) OVER (PARTITION BY pagos_diarios_mensuales.id_producto, pagos_diarios_mensuales.id_corresponsal ORDER BY fecha_diaria) AS variacion_diaria,
    total_pagado_mensual - LAG(total_pagado_mensual) OVER (PARTITION BY pagos_diarios_mensuales.id_producto, pagos_diarios_mensuales.id_corresponsal ORDER BY fecha_mensual) AS variacion_mensual
FROM pagos_diarios_mensuales
JOIN dim_producto pr ON pagos_diarios_mensuales.id_producto = pr.id_producto
JOIN dim_corresponsal co ON pagos_diarios_mensuales.id_corresponsal = co.id_corresponsal
ORDER BY pr.nombre_producto, co.nombre_corresponsal, fecha_diaria;
"""
df_variacion_diaria_mensual_productos_corresponsal = pd.read_sql(consulta_variacion_diaria_mensual_productos_corresponsal, conn)
df_variacion_diaria_mensual_productos_corresponsal.sort_values(by=["nombre_corresponsal", "nombre_producto", "fecha_diaria"], inplace=True)
print(df_variacion_diaria_mensual_productos_corresponsal)

                   nombre_producto         nombre_corresponsal fecha_diaria  \
0                ALTAVOZ SONY XB33                 AUDIO MUNDO   2025-04-29   
1   AURICULARES INALAMBRICOS NTUNE                 AUDIO MUNDO   2025-04-29   
7      LUCES LED INTERIORES   AUTO                 AUDIO MUNDO   2025-04-29   
8           MICROFONO USB C NSOUND                 AUDIO MUNDO   2025-04-29   
2   AURICULARES INALAMBRICOS NTUNE                   AUTO ZONA   2025-04-30   
3          CAMARA DE REVERSA NTECH                   AUTO ZONA   2025-04-29   
11       RADIO DE AUTO PIONEER DEH                   AUTO ZONA   2025-04-30   
14    SENSOR DE PARQUEO   4 PUNTOS                   AUTO ZONA   2025-04-29   
9        PANTALLA TACTIL 7"   AUTO               ELECTRO HOGAR   2025-04-29   
12       RADIO DE AUTO PIONEER DEH               ELECTRO HOGAR   2025-04-29   
17     SMARTWATCH XIAOMI MI BAND 7               ELECTRO HOGAR   2025-04-30   
20            TELEVISOR LED 50" 4K               ELE

/tmp/ipykernel_1554/277959218.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_variacion_diaria_mensual_productos_corresponsal = pd.read_sql(consulta_variacion_diaria_mensual_productos_corresponsal, conn)


In [35]:
conn.close()